# 01 — Prepare Data (local files → cache)

**Mục đích:** đọc `Datathon_Data/train/` & `Datathon_Data/test/test/` từ ổ đĩa local,
cache 6 file parquet xuống `cache_drive/`. Mọi notebook sau train từ cache.

**Setup trước khi chạy:**
1. Đảm bảo dữ liệu nằm ở `d:/Datathon_2/Datathon_Data/{train,test}/`.
2. Kernel Python local đã cài: `pyarrow pandas numpy scipy lightgbm scikit-learn tqdm`.

Output files trong `d:/Datathon_2/cache_drive/`:
- `dim_listing.parquet` (FULL, ~50MB)
- `snapshot_60d.parquet` (last 60d)
- `pci_full.parquet`
- `events_positive.parquet` (5 event_type, date ≤ 2026-04-09, ~96M row)
- `events_pageview_30d.parquet`
- `test_users.parquet`

In [ ]:
# Local kernel: assume deps already installed.
# To install run once:
#   pip install pyarrow pandas numpy scipy lightgbm scikit-learn tqdm
print("Skipping pip install (local kernel).")

In [ ]:
# === Local setup: paths + add training/utils to sys.path ===
import os, sys

PROJECT_DIR  = r"d:/Datathon_2"
TRAINING_DIR = os.path.join(PROJECT_DIR, "training")
CACHE_DIR    = os.path.join(PROJECT_DIR, "cache_drive")
DATA_DIR     = os.path.join(PROJECT_DIR, "Datathon_Data")  # contains train/ and test/
os.makedirs(CACHE_DIR, exist_ok=True)

if TRAINING_DIR not in sys.path:
    sys.path.insert(0, TRAINING_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR   :", DATA_DIR)
print("CACHE_DIR  :", CACHE_DIR)
print("utils available:", os.path.isdir(os.path.join(TRAINING_DIR, "utils")))

In [ ]:
TRAIN_DATE_START = "2025-11-09"
TRAIN_DATE_END = "2026-04-09"
VALID_START = "2026-03-13"

POSITIVE_EVENT_TYPES = frozenset({
    "view_phone", "contact_chat", "other_interaction", "contact_zalo", "contact_sms",
})
HIGH_INTENT_EVENTS = frozenset({"view_phone", "contact_chat", "contact_zalo", "contact_sms"})

INTENT_WEIGHT = {
    "view_phone": 3.0, "contact_chat": 2.0,
    "contact_zalo": 2.0, "contact_sms": 2.0,
    "other_interaction": 1.0,
}

# Local data paths (relative to DATA_DIR defined in the setup cell)
TRAIN_PATH = os.path.join(DATA_DIR, "train") + os.sep
TEST_PATH  = os.path.join(DATA_DIR, "test", "test") + os.sep

print("Constants loaded. TRAIN_END:", TRAIN_DATE_END, "VALID_START:", VALID_START)
print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH :", TEST_PATH)

In [ ]:
# Local kernel: no auth needed.
import os
assert os.path.isdir(TRAIN_PATH), f"Missing train dir: {TRAIN_PATH}"
assert os.path.isfile(os.path.join(TEST_PATH, "test_users.parquet")), \
    f"Missing test_users.parquet under {TEST_PATH}"
print("Local data OK.")

In [ ]:
import time
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds

t_start = time.time()

dim_dset  = ds.dataset(os.path.join(TRAIN_PATH, "dim_listing"),                    format="parquet")
snap_dset = ds.dataset(os.path.join(TRAIN_PATH, "fact_listing_snapshot"),          format="parquet")
pci_dset  = ds.dataset(os.path.join(TRAIN_PATH, "fact_post_contact_interactions"), format="parquet")
fue_dset  = ds.dataset(os.path.join(TRAIN_PATH, "fact_user_events"),               format="parquet")

train_end_d = pa.scalar(pd.Timestamp(TRAIN_DATE_END).date(), type=pa.date32())
snap_start_d = pa.scalar(pd.Timestamp("2026-02-09").date(), type=pa.date32())
pv_start_d   = pa.scalar(pd.Timestamp("2026-03-10").date(), type=pa.date32())

print(f"dim files     : {len(dim_dset.files):,}")
print(f"snap files    : {len(snap_dset.files):,}")
print(f"pci files     : {len(pci_dset.files):,}")
print(f"events files  : {len(fue_dset.files):,}")

In [ ]:
# ---- 0) Print schema for each dataset (verify before reading) ----
def _safe_cols(dset, wanted):
    have = set(dset.schema.names)
    keep = [c for c in wanted if c in have]
    miss = [c for c in wanted if c not in have]
    if miss:
        print(f"  [warn] missing cols dropped: {miss}")
    return keep

for name, dset in [("dim_listing", dim_dset), ("fact_listing_snapshot", snap_dset),
                   ("fact_post_contact_interactions", pci_dset),
                   ("fact_user_events", fue_dset)]:
    print(f"\n=== {name} schema ===")
    print(dset.schema)

In [ ]:
# ---- 1) dim_listing FULL ----
DIM_COLS_WANT = [
    "item_id", "seller_id", "category", "title",
    "seller_type", "ad_type", "ad_status",
    "area_sqm", "bedrooms", "bathrooms", "floors", "width_m",
    "direction", "legal_status", "house_type", "furnishing",
    "city_name", "district_name", "ward_name", "project_id",
    "price_bucket", "images_count", "posted_date", "expected_expired_date",
]
DIM_COLS = _safe_cols(dim_dset, DIM_COLS_WANT)
t0 = time.time()
dim_tbl = dim_dset.to_table(columns=DIM_COLS)
df_dim = dim_tbl.to_pandas()
print(f"dim_listing: {df_dim.shape} | {time.time()-t0:.1f}s")
print(f"posted_date: {df_dim['posted_date'].min()} -> {df_dim['posted_date'].max()}")
print(df_dim["ad_status"].value_counts(dropna=False).head(10))

df_dim.to_parquet(f"{CACHE_DIR}/dim_listing.parquet", index=False)
del dim_tbl, df_dim

In [ ]:
# ---- 2) fact_listing_snapshot last 60d (streaming write, 8GB-safe) ----
import pyarrow.parquet as pq
flt_snap = (pc.field("date") >= snap_start_d) & (pc.field("date") <= train_end_d)
SNAP_COLS_WANT = ["item_id", "date", "views_24h", "contacts_24h",
                  "listing_age_days", "category"]
SNAP_COLS = _safe_cols(snap_dset, SNAP_COLS_WANT)
t0 = time.time()
out_snap = f"{CACHE_DIR}/snapshot_60d.parquet"
writer = None
n_rows = 0
for batch in snap_dset.scanner(columns=SNAP_COLS, filter=flt_snap,
                                batch_size=500_000).to_batches():
    if writer is None:
        writer = pq.ParquetWriter(out_snap, batch.schema, compression="snappy")
    writer.write_batch(batch)
    n_rows += batch.num_rows
if writer is not None:
    writer.close()
print(f"snapshot_60d: {n_rows:,} rows | {time.time()-t0:.1f}s -> {out_snap}")

In [ ]:
# ---- 3) fact_post_contact_interactions FULL (date <= TRAIN_END, streaming) ----
flt_pci = pc.field("date") <= train_end_d
PCI_COLS_WANT = ["user_id", "item_id", "date", "category", "purchased"]
PCI_COLS = _safe_cols(pci_dset, PCI_COLS_WANT)
t0 = time.time()
out_pci = f"{CACHE_DIR}/pci_full.parquet"
writer = None
n_rows = 0
for batch in pci_dset.scanner(columns=PCI_COLS, filter=flt_pci,
                               batch_size=500_000).to_batches():
    if writer is None:
        writer = pq.ParquetWriter(out_pci, batch.schema, compression="snappy")
    writer.write_batch(batch)
    n_rows += batch.num_rows
if writer is not None:
    writer.close()
print(f"pci_full: {n_rows:,} rows | {time.time()-t0:.1f}s -> {out_pci}")

In [ ]:
# ---- 4) fact_user_events POSITIVE only (5 event_type, date <= TRAIN_END) ----
pos_arr = pa.array(sorted(POSITIVE_EVENT_TYPES))
flt_pos = (
    (pc.field("date") <= train_end_d)
    & pc.is_in(pc.field("event_type"), value_set=pos_arr)
)
POS_COLS_WANT = ["user_id", "session_id", "item_id", "event_type", "event_ts",
                 "date", "category", "surface", "device", "dwell_time_sec",
                 "is_login", "city_name"]
POS_COLS = _safe_cols(fue_dset, POS_COLS_WANT)
t0 = time.time()
import pyarrow.parquet as pq
out_path_pos = f"{CACHE_DIR}/events_positive.parquet"
writer = None
n_rows = 0
for batch in fue_dset.scanner(
    columns=POS_COLS, filter=flt_pos, batch_size=400_000
).to_batches():
    if writer is None:
        writer = pq.ParquetWriter(out_path_pos, batch.schema, compression="snappy")
    writer.write_batch(batch)
    n_rows += batch.num_rows
if writer is not None:
    writer.close()
print(f"events_positive: {n_rows:,} rows | {time.time()-t0:.1f}s -> {out_path_pos}")

In [ ]:
# ---- 5) fact_user_events PAGEVIEW last 30d ----
flt_pv = (
    (pc.field("date") >= pv_start_d)
    & (pc.field("date") <= train_end_d)
    & (pc.field("event_type") == pa.scalar("pageview"))
)
PV_COLS_WANT = ["user_id", "item_id", "event_ts", "date",
                "dwell_time_sec", "is_login"]
PV_COLS = _safe_cols(fue_dset, PV_COLS_WANT)
t0 = time.time()
out_path_pv = f"{CACHE_DIR}/events_pageview_30d.parquet"
writer = None
n_rows = 0
for batch in fue_dset.scanner(
    columns=PV_COLS, filter=flt_pv, batch_size=400_000
).to_batches():
    if writer is None:
        writer = pq.ParquetWriter(out_path_pv, batch.schema, compression="snappy")
    writer.write_batch(batch)
    n_rows += batch.num_rows
if writer is not None:
    writer.close()
print(f"events_pageview_30d: {n_rows:,} rows | {time.time()-t0:.1f}s")

In [ ]:
# ---- 6) test_users ----
df_test = pd.read_parquet(os.path.join(TEST_PATH, "test_users.parquet"))
print(f"test_users: {len(df_test):,}")
df_test.to_parquet(f"{CACHE_DIR}/test_users.parquet", index=False)
print(f"\nTOTAL ELAPSED: {time.time()-t_start:.0f}s")

In [ ]:
# ---- Verification: confirm all 6 cache files exist + sane ----
import os
for name in ["dim_listing.parquet", "snapshot_60d.parquet", "pci_full.parquet",
             "events_positive.parquet", "events_pageview_30d.parquet",
             "test_users.parquet"]:
    p = os.path.join(CACHE_DIR, name)
    sz = os.path.getsize(p) / (1024**2)
    print(f"  {name:35s} {sz:8.1f} MB")

df_t = pd.read_parquet(f"{CACHE_DIR}/test_users.parquet")
assert len(df_t) == 161_568, f"test_users mismatch: {len(df_t)}"
df_p = pd.read_parquet(f"{CACHE_DIR}/events_positive.parquet", columns=["event_type", "date"])
assert df_p["event_type"].isin(POSITIVE_EVENT_TYPES).all()
assert pd.to_datetime(df_p["date"]).max().date() <= pd.Timestamp(TRAIN_DATE_END).date()
print("\nAll assertions passed. Cache ready for notebooks 02-07.")